# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 9.30e-14 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.00e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405022e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 5.21898e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797539388882393 - 0.09754482372919099*y - 0.4175945425725998*x + 2.1931519300021916*y^2 - 0.1269608752480836*x*y - 0.8025868707701475*x^2)^2 + (0.9043585861943056 - 0.6454975493868688*y + 0.2742173500241793*x - 0.10770166105288279*y^2 - 1.2293131412594325*x*y - 0.9300170435472996*x^2)^2 + (-0.5739156167209173 - 1.3853512632170424*y - 0.6219655290929101*x - 0.20409240982196922*y^2 - 0.15182435477475986*x*y + 0.4443814867923305*x^2)^2 + (0.5703017871356986 - 0.5184102408490359*y + 0.34367412466167085*x + 0.2555951335437905*y^2 + 0.7618448416641994*x*y - 0.02090567085116562*x^2)^2 + (0.04140298207812761 + 0.04911089940832097*y - 0.49990900774527314*x - 0.24792082949867686*y^2 + 0.2972448439664639*x*y - 0.5054176772471584*x^2)^2 + (-0.25245328219549756 - 0.06336616977802403*y + 0.25394133757830056*x - 0.10018803519680765*y^2 + 0.05960215579665097*x*y - 0.1938133953768724*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Computing a Gröbner basis before passing such a domain to SumOfSquares
can take a non-negligible amount of time for large systems of equalities.
So this is not done by default in SumOfSquares but we can request it
explicitly with `SemialgebraicSets.compute_gröbner_basis!`.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  149.956 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  292.911 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
To reduce the polynomial modulo the full ideal, we compute the Gröbner
basis before passing the domain to the SOS constraint.

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
SemialgebraicSets.compute_gröbner_basis!(S.I)
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 1.44e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 1.79e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.64e-01 Pobj:  0.0000000e+00 Ad: 8.07e-01 Dobj:  2.6104524e+00 
Iter:  2 Ap: 7.71e-01 Pobj:  0.0000000e+00 Ad: 7.40e-01 Dobj: -5.9089088e-02 
Iter:  3 Ap: 8.08e-01 Pobj:  0.0000000e+00 Ad: 7.65e-01 Dobj:  5.5548423e-01 
Iter:  4 Ap: 7.56e-01 Pobj:  0.0000000e+00 Ad: 7.79e-01 Dobj:  4.8507215e-01 
Iter:  5 Ap: 6.95e-01 Pobj:  0.0000000e+00 Ad: 6.77e-01 Dobj:  4.3029745e-01 
Iter:  6 Ap: 6.35e-01 Pobj:  0.0000000e+00 Ad: 6.83e-01 Dobj:  2.2138618e-01 
Iter:  7 Ap: 6.21e-01 Pobj:  0.0000000e+00 Ad: 6.43e-01 Dobj:  1.2987284e-01 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 2.77281e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-5.120335179072195e-5 + 0.001057995101415955*y + 0.000340676045802126*x + 8.156734411468465*y^2 - 32.77236925893393*x*y + 29.43238259971488*x^2)^2 + (-1.0587274914098373e-5 + 0.00010594598825375375*y + 0.0010731384717902641*x + 26.095043463954315*y^2 - 16.091430394680042*x*y - 25.149327811316386*x^2)^2 + (2.9699690140733147e-5 + 0.0007942841656529818*y - 0.00016631192873664265*x - 0.12254840671007258*y^2 - 0.09189396898532246*x*y - 0.0683597073232651*x^2)^2 + (-5.173506174375948e-7 - 0.03720819599236971*y + 0.049792889135531135*x - 0.00016552561054185747*y^2 - 0.00012343252090798703*x*y - 9.080559085539111e-5*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

3.616212870828050010933800881523896009195340184305678121745586395263671875e-9 + 7.859383493666598680203114302323115681066412481670387280293000050080422986348211e-8y + 5.933210702234139087242790712760044978215144230769230769230769230769230769230765e-8x - 8.922894778329319690379946294900292969032307155430316925048828125e-6y² - 1.3342315251522617299884988784075545709129073657095432281494140625e-5xy - 4.9808615240888128969007873914076611754353507421910762786865234375e-6x² - 6.616660878459394249754647413889567057291666666666666666666666666666671272623229e-8y³ - 7.105253022980188385796888420979181925455729166666666666666666666666664363688385e-8xy² - 5.8576475944454504973281372404159128564060665667057037353515625e-8x²y + 5.504611740784289745184091421274038461538461538461538461538461538461538461538778e-9y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

4.2821992984688214184420048812225456646984866893035359680652618408203125e-9 + 1.554808257886934910808715844432744035704019598853893769093048878209441860601999e-9y + 1.813330887155363877984480215952946589543269230769230769230769230769913690867597e-9x + 4.28220150289725098868842678712098859250545501708984375e-9y² + 8.1466951240560803171320003457367420196533203125e-15xy + 4.2821985614567570355148973249015398323535919189453125e-9x² + 5.921189464667501548926035563151041666666666666666666666666666643636883853081481e-15y³ + 4.233650467237263607482115427652994791666666666666666666666666667818155807345926e-14xy² + 7.88535903240017432835884392261505126953125e-14x²y + 5.675736666979411473641028771033653846153846153846153846153846153840690473061801e-9y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*